# Collaborative filtering — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x)))
def dcg(rels): return sum((2**r-1)/math.log2(i+2) for i,r in enumerate(rels))
def ndcg(rels):
    best=sorted(rels, reverse=True)
    return dcg(rels)/dcg(best) if dcg(best)>0 else 0.0
print('setup ok')

## Predicting from neighbors in the user-item matrix
Collaborative filtering starts from the sparse rating matrix and borrows evidence from similar users or similar items. These five examples compute a user-user prediction, expose the centered version, compare item neighbors, show sparsity, and run a tiny leave-one-out check.

In [ ]:
# 1) user-user cosine neighbors predict Ava's missing rating for item 3
R=np.array([[5,4,0,1],[4,5,0,1],[1,1,5,4],[0,1,4,5]],float)
avail=np.array([False,False,True,True]); target=0; item=2
sims=np.array([cosine(R[target,[0,1,3]], R[u,[0,1,3]]) for u in range(4)])
neighbors=np.array([2,3]); weights=sims[neighbors]; ratings=R[neighbors,item]
pred=float(weights@ratings/weights.sum())
plt.figure(figsize=(6,3)); plt.bar(['u2 sim','u3 sim'],weights); plt.axhline(pred/5,c='r',ls='--'); plt.title(f'user-user weighted rating = {pred:.3f}')
assert abs(pred-4.634503748169032)<1e-9

In [ ]:
# 2) mean-centering removes generous-rating bias before averaging residuals
R=np.array([[5,4,0,1],[4,5,0,1],[5,4,5,1],[4,4,4,1]],float)
means=np.array([R[u,R[u]>0].mean() for u in range(4)])
centered=np.where(R>0,R-means[:,None],0)
sims=np.array([cosine(centered[0,[0,1,3]], centered[u,[0,1,3]]) for u in range(4)])
neighbors=np.array([2,3]); weights=np.maximum(sims[neighbors],0); residuals=R[neighbors,2]-means[neighbors]
pred=float(means[0] + weights@residuals/weights.sum())
plt.figure(figsize=(6,3)); plt.bar(['mean Ava','centered pred'],[means[0],pred]); plt.title(f'centered prediction = {pred:.3f}')
assert abs(pred-4.335323007819495)<1e-9

In [ ]:
# 3) item-item collaborative filtering uses similar items instead of similar users
R=np.array([[5,4,0,1],[4,5,0,1],[1,1,5,4],[0,1,4,5]],float)
item=2; user=0; rated=np.where(R[user]>0)[0]
sims=[]
for j in rated:
    both=(R[:,item]>0)&(R[:,j]>0)
    sims.append(cosine(R[both,item],R[both,j]))
sims=np.array(sims); pred=float(sims@R[user,rated]/sims.sum())
plt.figure(figsize=(6,3)); plt.bar([f'i{j}' for j in rated],sims); plt.title(f'item-item prediction = {pred:.3f}')
assert abs(pred-3.351125276320345)<1e-9

In [ ]:
# 4) sparsity: fewer overlaps make similarities unstable
R=np.array([[5,4,0,1],[4,5,0,1],[1,1,5,4],[0,1,4,5]],float)
overlap=np.zeros((4,4),int)
for a in range(4):
    for b in range(4): overlap[a,b]=int(((R[a]>0)&(R[b]>0)).sum())
plt.figure(figsize=(4,4)); plt.imshow(overlap,cmap='Blues',vmin=0,vmax=4); plt.colorbar(label='co-rated items'); plt.title('overlap counts')
assert overlap[0,3]==2 and overlap[0,2]==3

In [ ]:
# 5) leave-one-out check on a tiny matrix
R=np.array([[5,4,0,1],[4,5,0,1],[1,1,5,4],[0,1,4,5]],float)
pairs=[(0,1),(1,0),(2,3),(3,2)]; errs=[]
for u,i in pairs:
    T=R.copy(); true=T[u,i]; T[u,i]=0
    users=[v for v in range(4) if v!=u and T[v,i]>0]
    sims=np.array([cosine(T[u,T[u]>0], T[v,T[u]>0]) for v in users])
    p=float(sims@T[users,i]/sims.sum()); errs.append(abs(p-true))
plt.figure(figsize=(6,3)); plt.bar([str(p) for p in pairs],errs); plt.ylabel('absolute error'); plt.title(f'MAE = {np.mean(errs):.3f}')
assert abs(float(np.mean(errs))-0.5402231807117724)<1e-9